# 02 · Feature Engineering
**Epidemiological Pulse – Population Health Hotspot Detection**

This notebook demonstrates the complete feature engineering pipeline.
We will:
1. Load and preprocess the master dataset
2. Apply each feature builder in sequence
3. Inspect and visualise the resulting features
4. Examine the composite risk score


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from src.data.preprocess import PreprocessingPipeline
from src.features.build_features import FeatureEngineeringPipeline

DATA_DIR = Path('../data/synthetic')
print('Ready.')

## 1 · Load & Preprocess

In [ ]:
import yaml
with open('../config/params.yaml') as f:
    config = yaml.safe_load(f)

preprocess_pipeline = PreprocessingPipeline(config)

# Load individual CSVs
raw = {
    'pharmacy':   pd.read_csv(DATA_DIR / 'pharmacy_sales.csv',        parse_dates=['date']),
    'sentiment':  pd.read_csv(DATA_DIR / 'social_media_sentiment.csv', parse_dates=['date']),
    'weather':    pd.read_csv(DATA_DIR / 'weather.csv',               parse_dates=['date']),
    'aqi':        pd.read_csv(DATA_DIR / 'aqi.csv',                   parse_dates=['date']),
    'er_visits':  pd.read_csv(DATA_DIR / 'er_visits.csv',             parse_dates=['date']),
}

df_clean, train, val, test = preprocess_pipeline.run(raw)
print(f'Clean shape: {df_clean.shape}')
print(f'Train: {train.shape}  Val: {val.shape}  Test: {test.shape}')

## 2 · Run Feature Engineering Pipeline

In [ ]:
fe_pipeline = FeatureEngineeringPipeline(config)
df_features = fe_pipeline.run(df_clean)

print(f'Feature matrix shape: {df_features.shape}')
print(f'\nNew columns added ({df_features.shape[1] - df_clean.shape[1]}):')
new_cols = [c for c in df_features.columns if c not in df_clean.columns]
for col in new_cols:
    print(f'  {col}')

## 3 · Temporal Features

In [ ]:
temporal_cols = [c for c in df_features.columns if any(
    x in c for x in ['dow', 'month', 'season', 'sin', 'cos', 'is_weekend']
)]
print('Temporal features:', temporal_cols)

# Check cyclical encoding
sample = df_features[['date', 'day_of_week', 'dow_sin', 'dow_cos']].drop_duplicates('day_of_week')
display(sample.sort_values('day_of_week'))

## 4 · Lag & Rolling Features

In [ ]:
zip_code = df_features['zip_code'].unique()[0]
df_zip = df_features[df_features['zip_code'] == zip_code].sort_values('date').copy()

lag_cols   = [c for c in df_zip.columns if 'lag' in c and 'pharmacy' in c]
roll_cols  = [c for c in df_zip.columns if 'roll' in c and 'pharmacy' in c]

print(f'Pharmacy lag features:    {lag_cols}')
print(f'Pharmacy rolling features: {roll_cols}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['pharmacy_sales'],
                         name='Original', line=dict(color='steelblue')))
for col in roll_cols[:3]:
    fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip[col], name=col))

fig.update_layout(title='Pharmacy Sales: Original vs Rolling Features',
                  template='plotly_dark')
fig.show()

## 5 · Anomaly Features

In [ ]:
anomaly_cols = [c for c in df_zip.columns if 'zscore' in c or 'anomaly' in c]
print('Anomaly features:', anomaly_cols)

if 'pharmacy_sales_zscore' in df_zip.columns:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Pharmacy Sales', 'Z-Score'])
    fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['pharmacy_sales'],
                             name='Sales'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['pharmacy_sales_zscore'],
                             name='Z-Score', line=dict(color='orange')), row=2, col=1)
    fig.add_hline(y=2.5, line_dash='dash', line_color='red', row=2, col=1)
    fig.add_hline(y=-2.5, line_dash='dash', line_color='red', row=2, col=1)
    fig.update_layout(height=600, template='plotly_dark')
    fig.show()

## 6 · Composite Risk Score

In [ ]:
if 'risk_score' in df_features.columns:
    fig = px.histogram(
        df_features, x='risk_score', color='zip_code',
        nbins=50, barmode='overlay',
        title='Distribution of Composite Risk Score by ZIP Code',
        template='plotly_dark', opacity=0.7
    )
    fig.add_vline(x=0.33, line_dash='dash', line_color='yellow',
                  annotation_text='Low/Med')
    fig.add_vline(x=0.66, line_dash='dash', line_color='red',
                  annotation_text='Med/High')
    fig.show()

    # Time-series of risk score for one ZIP
    fig2 = px.line(
        df_zip, x='date', y='risk_score',
        title=f'Risk Score Over Time – ZIP {zip_code}',
        template='plotly_dark'
    )
    fig2.add_hline(y=0.66, line_dash='dash', line_color='red', annotation_text='High risk')
    fig2.show()
else:
    print('risk_score column not found – check config weights')

## 7 · Feature Importance (Correlation with ER Visits)

In [ ]:
target = 'er_visits'
if target in df_features.columns:
    numeric_df = df_features.select_dtypes(include=[np.number])
    corr_with_target = (
        numeric_df.corr()[target]
        .drop(target)
        .abs()
        .sort_values(ascending=False)
        .head(20)
    )

    fig = px.bar(
        corr_with_target.reset_index(),
        x='index', y=target,
        title='Top 20 Features by |Correlation| with ER Visits',
        labels={'index': 'Feature', target: '|Correlation|'},
        color=target, color_continuous_scale='Reds',
        template='plotly_dark'
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

print('\n✅ Feature engineering notebook complete.')